# Final Model 1 - STREAMLIT

In [ ]:
# Cell 1: Import Libraries and Mount Google Drive
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Cell 2: Load the Dataset & Clean the 'Complications' Column

# Update the path to match where your Data1.xlsx file is located
file_path = r"/content/drive/My Drive/Yr5/Cuatri2/TFG: Capstone project/Data1.xlsx"

# Load the dataset
data = pd.read_excel(file_path)

# Replace "None" or blank entries with a real label "No complications"
data['Complications'] = data['Complications'].fillna("No complications")
data['Complications'] = data['Complications'].replace("None", "No complications")

# Show a preview of the data
print("Data preview:")
print(data.head(10))

Data preview:
   Patient number  Gender  Age        Operation        Method  \
0               1    Male   45  Cholecystectomy  Laparoscopic   
1               2    Male   58  Cholecystectomy  Laparoscopic   
2               3  Female   43  Cholecystectomy  Laparoscopic   
3               4  Female   11  Cholecystectomy  Laparoscopic   
4               5    Male   18  Cholecystectomy  Laparoscopic   
5               6    Male   29  Cholecystectomy  Laparoscopic   
6               7  Female    9  Cholecystectomy  Laparoscopic   
7               8  Female   74  Cholecystectomy  Laparoscopic   
8               9    Male   58  Cholecystectomy  Laparoscopic   
9              10  Female   60  Cholecystectomy  Laparoscopic   

      Complications  Hospital stay (days) Surgery type  ASA Score  
0  No complications                     1       Urgent          2  
1  No complications                     0       Urgent          2  
2  No complications                     0      Planned          2 

In [ ]:
# Cell 3: Label-Encode the 'Complications' Column
from sklearn.preprocessing import LabelEncoder

# Create a LabelEncoder for complications
le_comp = LabelEncoder()
data['Complications'] = le_comp.fit_transform(data['Complications'])

# Print the text labels that were learned
print("Complication classes:", le_comp.classes_)


Complication classes: ['Bleeding' 'Infection' 'No complications' 'Organ damage' 'Reoperation']


In [ ]:
# Cell 4: Label-Encode Other Categorical Columns

categorical_cols = ['Gender', 'Operation', 'Method', 'Surgery type']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

# Let's see a preview
print("Preview after label-encoding categorical inputs:")
print(data.head(10))


Preview after label-encoding categorical inputs:
   Patient number  Gender  Age  Operation  Method  Complications  \
0               1       1   45          0       0              2   
1               2       1   58          0       0              2   
2               3       0   43          0       0              2   
3               4       0   11          0       0              2   
4               5       1   18          0       0              3   
5               6       1   29          0       0              0   
6               7       0    9          0       0              2   
7               8       0   74          0       0              2   
8               9       1   58          0       0              2   
9              10       0   60          0       0              2   

   Hospital stay (days)  Surgery type  ASA Score  
0                     1             1          2  
1                     0             1          2  
2                     0             0          2 

In [ ]:
# Cell 5: Define Features, Targets & Split Data
from sklearn.model_selection import train_test_split

# The columns used as features
features = ['Age', 'Gender', 'Operation', 'Method', 'Surgery type', 'ASA Score']
X = data[features]

# The classification target (complication category)
y_class = data['Complications']

# The regression target (hospital stay in days)
y_reg = data['Hospital stay (days)']

# Split into training and testing sets
X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)


Training set size: (800, 6)
Testing set size: (200, 6)


In [ ]:
# Cell 6: Train the Classification Model (Complications)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_class_train)

# Evaluate on the test set
y_class_pred = clf.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_class_test, y_class_pred))



Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         6
           1       0.36      0.54      0.43        28
           2       0.84      0.81      0.83       154
           3       0.00      0.00      0.00         4
           4       0.33      0.12      0.18         8

    accuracy                           0.70       200
   macro avg       0.31      0.29      0.29       200
weighted avg       0.71      0.70      0.70       200



In [ ]:
# Cell 7: Train the Regression Model (Hospital Stay)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

reg = RandomForestRegressor(random_state=42)
reg.fit(X_train, y_reg_train)

# Evaluate on the test set
y_reg_pred = reg.predict(X_test)
print("\nRegression Metrics for Hospital Stay:")
print("Mean Absolute Error:", mean_absolute_error(y_reg_test, y_reg_pred))
print("R2 Score:", r2_score(y_reg_test, y_reg_pred))



Regression Metrics for Hospital Stay:
Mean Absolute Error: 1.0374968055555556
R2 Score: 0.8050063190626444


In [ ]:
# Cell 8: Define a Prediction Function for New Inputs

def predict_outcomes(age, gender, operation, method, surgery_type, asa_score):
    """
    Given patient inputs, returns:
      1) A dictionary with predicted probabilities for each complication class.
      2) The estimated number of days for hospital stay.
    """
    import pandas as pd

    # Create a DataFrame for a single input
    input_data = pd.DataFrame({
        'Age': [age],
        'Gender': [gender],
        'Operation': [operation],
        'Method': [method],
        'Surgery type': [surgery_type],
        'ASA Score': [asa_score]
    })

    # Convert each categorical input to numeric using the stored label encoders
    for col in ['Gender', 'Operation', 'Method', 'Surgery type']:
        input_data[col] = label_encoders[col].transform(input_data[col])

    # Predict complication probabilities
    comp_prob = clf.predict_proba(input_data)

    # Map numeric class indices back to string labels (e.g., "Bleeding", "No complications", etc.)
    comp_mapping = {i: label for i, label in enumerate(le_comp.classes_)}
    prob_dict = {comp_mapping[i]: comp_prob[0][i] for i in range(len(comp_prob[0]))}

    # Predict hospital stay (days)
    los = reg.predict(input_data)[0]

    return prob_dict, los

print("Prediction function defined successfully!")


Prediction function defined successfully!


In [ ]:
# Cell 9: Test the Prediction Function with an Example

example_input = {
    'age': 45,
    'gender': 'Male',              # Must match what your original data had before label encoding
    'operation': 'Cholecystectomy',
    'method': 'Laparoscopic',
    'surgery_type': 'Planned',
    'asa_score': 2
}

# Get predictions
probs, estimated_los = predict_outcomes(**example_input)

print("Predicted Complication Probabilities:")
for complication, prob in probs.items():
    print(f"{complication}: {prob*100:.1f}%")

print("\nEstimated Hospital Stay:", estimated_los, "days")


Predicted Complication Probabilities:
Bleeding: 0.0%
Infection: 1.0%
No complications: 99.0%
Organ damage: 0.0%
Reoperation: 0.0%

Estimated Hospital Stay: 1.63 days


In [ ]:
# Cell 8: Interactive Prediction UI

import ipywidgets as widgets
from IPython.display import display

# Create input widgets
age_widget = widgets.IntText(value=45, description="Age:")
gender_widget = widgets.Dropdown(options=["Male", "Female"], description="Gender:")
operation_widget = widgets.Dropdown(options=["Cholecystectomy", "Hemicolectomy"], description="Operation:")
method_widget = widgets.Dropdown(options=["Laparoscopic", "Open"], description="Method:")
surgery_type_widget = widgets.Dropdown(options=["Planned", "Urgent"], description="Surgery Type:")
asa_score_widget = widgets.IntSlider(value=2, min=1, max=10, step=1, description="ASA Score:")

# Create a button to trigger prediction
predict_button = widgets.Button(description="Predict")

# Create an output area
output_area = widgets.Output()

# Define what happens when we click the "Predict" button
def on_predict_button_click(b):
    with output_area:
        # Clear previous output
        output_area.clear_output()

        # Collect user input
        age = age_widget.value
        gender = gender_widget.value
        operation = operation_widget.value
        method = method_widget.value
        surgery_type = surgery_type_widget.value
        asa_score = asa_score_widget.value

        # Make the prediction
        probs, estimated_los = predict_outcomes(age, gender, operation, method, surgery_type, asa_score)

        # Print results
        print("Predicted Complication Probabilities:")
        for complication, prob in probs.items():
            print(f"{complication}: {prob*100:.1f}%")
        print("\nEstimated Hospital Stay:", estimated_los, "days")

# Attach the click event to our button
predict_button.on_click(on_predict_button_click)

# Display all widgets and the output area
display(
    widgets.VBox([
        age_widget,
        gender_widget,
        operation_widget,
        method_widget,
        surgery_type_widget,
        asa_score_widget,
        predict_button,
        output_area
    ])
)

In [ ]:
import pickle

# Save the classifier, regressor, complication encoder, and input label encoders into one file
models = {
    "clf": clf,
    "reg": reg,
    "le_comp": le_comp,
    "label_encoders": label_encoders
}

with open("models.pkl", "wb") as f:
    pickle.dump(models, f)

print("Models saved to models.pkl")


Models saved to models.pkl


In [ ]:
import sklearn
print(sklearn.__version__)


1.6.1


In [ ]:
%%writefile app.py
import streamlit as st, pandas as pd, pickle

# --- CUSTOM CSS ---
st.markdown("""
    <style>
    /* Page background + all labels/titles in white */
    .stApp {
        background-color: #40E0D0;
    }
    h1, .stMarkdown, label {
        color: #ffffff !important;
    }

    /* Input‑box text (number & text fields) */
    .stTextInput input, .stNumberInput input {
        color: #888 !important;
    }

    /* Selectbox selected value + placeholder */
    .stSelectbox .css-1uccc91-singleValue,
    .stSelectbox .css-14el2xx-placeholder {
        color: #888 !important;
    }

    /* Dropdown menu items */
    .stSelectbox .css-26l3qy-menu div {
        color: #888 !important;
    }

    /* Slider value */
    .stSlider > div > input {
        color: #888 !important;
    }

    /* Predict button */
    div.stButton > button {
        background-color: #ff0000 !important;
        color: #ffffff !important;
        border-radius: 4px;
        border: none;
    }
    </style>
""", unsafe_allow_html=True)



# --- LOAD MODELS ---
with open("models.pkl","rb") as f:
    m = pickle.load(f)
clf, reg, le_comp, encoders = m["clf"], m["reg"], m["le_comp"], m["label_encoders"]

# --- PREDICTION FUNCTION ---
def predict_outcomes(age, gender, operation, method, surgery_type, asa_score):
    df = pd.DataFrame({
        'Age':[age], 'Gender':[gender],
        'Operation':[operation], 'Method':[method],
        'Surgery type':[surgery_type], 'ASA Score':[asa_score]
    })
    for c, le in encoders.items():
        df[c] = le.transform(df[c])
    probs = clf.predict_proba(df)[0]
    mapping = {i:lab for i,lab in enumerate(le_comp.classes_)}
    return {mapping[i]: probs[i] for i in range(len(probs))}, reg.predict(df)[0]

# --- STREAMLIT APP UI ---
st.title("Surgery Outcome Predictor")
age = st.number_input("Age", 0, 120, 45)
gender = st.selectbox("Gender", ["Male","Female"])
operation = st.selectbox("Operation", ["Cholecystectomy","Hemicolectomy"])
method = st.selectbox("Method", ["Laparoscopic","Open"])
surgery_type = st.selectbox("Surgery Type", ["Planned","Urgent"])
asa_score = st.slider("ASA Score", 1, 5, 2)

if st.button("Predict"):
    probs, los = predict_outcomes(age, gender, operation, method, surgery_type, asa_score)
    st.subheader("Predicted Complication Probabilities:")
    for comp, prob in probs.items():
        st.write(f"{comp}: {prob*100:.1f}%")
    st.subheader("Estimated Hospital Stay:")
    st.write(f"{los:.1f} days")


Writing app.py
